# 知识架构系列 1：消失的上下文 —— RAG vs KV (CAG)
### *案例：GlobalCorp 跨境数据迁移项目的审计危机*

在大型企业中，规章制度通常由多个委员会编写，导致关键定义和处罚条款往往相隔几十页。  

**本章实验重点：**
1. **生成高度真实的 PDF**：模拟一份包含法律声明、行政杂项和深度业务定义的 15 页审计手册。
2. **逻辑穿透测试**：用户查询需要关联 **第 1 页的代号定义** 和 **第 15 页的财务罚则**。
3. **技术对比**：
    - **Vector RAG**：受限于分块大小（Chunk Size），通常只能看到局部，无法连接长距离逻辑。
    - **KV / CAG**：利用 1M 上下文窗口，实现对整份机密文件的“全脑感知”。

In [1]:
import os
from fpdf import FPDF
from pypdf import PdfReader
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

load_dotenv(override=True)

model = init_chat_model(
    model="agnes-2.0-flash",
    model_provider="openai",
    base_url=os.getenv("AGNES_BASE_URL"),
    api_key=os.getenv("AGNES_API_KEY"),
)

print("✅ 企业审计分析环境初始化完成。")

✅ 企业审计分析环境初始化完成。


## 1. 动态生成专业企业 PDF
我们将使用 `fpdf2` 生成一份排版精美、内容专业的内部文档。

In [2]:
def generate_polished_pdf(filename):
    pdf = FPDF()
    # 字体配置 (针对 macOS)
    font_path = "/System/Library/Fonts/Supplemental/Arial Unicode.ttf"
    local_font = "./Arial Unicode.ttf"
    if not os.path.exists(local_font): 
        import shutil; shutil.copy(font_path, local_font)
    
    pdf.add_font("CorpFont", "", local_font, uni=True)
    
    # --- 第 1 页：封面与核心定义 ---
    pdf.add_page()
    pdf.set_font("CorpFont", size=18)
    pdf.cell(200, 40, txt="GlobalCorp 战略基础设施内部规程", ln=True, align='C')
    pdf.set_font("CorpFont", size=10)
    pdf.cell(200, 10, txt="文档编号：GC-AUDIT-2025-099 | 密级：高度机密 (Internal Only)", ln=True, align='C')
    pdf.ln(20)
    
    pdf.set_font("CorpFont", size=14)
    pdf.cell(200, 10, txt="第一部分：核心业务定义与分类", ln=True)
    pdf.set_font("CorpFont", size=11)
    pdf.multi_cell(0, 8, txt=(
        "1.1 项目分类指南\n"
        "为了确保 GlobalCorp 在全球范围内的数据合规性，审计委员会定义了以下核心项目代号：\n\n"
        "【CYBER-TIGER (网络虎计划)】：本代号专指所有涉及跨国界（含大中华区与北美区）的所有云端数据迁移、"
        "数据库热备份及相关高延迟基础设施优化活动。所有标记为 CYBER-TIGER 的项目在操作前必须经过法律部审核。"
    ))

    # --- 第 2-14 页：大量的企业行政噪声 (模拟真实文档的复杂性) ---
    filler_titles = [
        "差旅与报销审批流程", "固定资产折旧计算标准", "办公室绿植租赁规范", 
        "供应商准入调查问卷", "内部会议纪要存档制度", "办公区空调能耗指标"
    ]
    for i in range(2, 15):
        pdf.add_page()
        pdf.set_font("CorpFont", size=12)
        title = filler_titles[(i-2) % len(filler_titles)]
        pdf.cell(200, 10, txt=f"第 {i} 章：{title}", ln=True)
        pdf.set_font("CorpFont", size=9)
        noise = ("根据集团 ESG 战略要求，所有员工应严格遵守相关流程，确保运营效率的持续优化。 " * 15 + "\n") * 10
        pdf.multi_cell(0, 6, txt=noise)

    # --- 第 15 页：深埋的红线规则 ---
    pdf.add_page()
    pdf.set_font("CorpFont", size=14)
    pdf.cell(200, 10, txt="第 99 章：合规性财务处罚细则", ln=True)
    pdf.ln(5)
    pdf.set_font("CorpFont", size=11)
    pdf.multi_cell(0, 8, txt=(
        "99.4 跨境业务特别违规处理\n"
        "针对所有涉及‘数据库热备份’或‘云端数据迁移’的违规操作，审计委员会设定了严格的红线。\n\n"
        "【重要处罚】：若发现该类活动在亚太地区 (APAC) 的分支机构存在未授权操作，"
        "相关事业部必须直接从当季度的利润中心中扣除该项目总预算的 35% 作为合规罚金。"
    ))

    pdf.output(filename)
    print(f"✅ 成功生成企业级专业审计手册: {filename}")

generate_polished_pdf("GlobalCorp_Professional_Audit.pdf")

/Users/weiping/dev/Learn/langchain-ai/deep_agent_0530/.venv/lib/python3.13/site-packages/fpdf/ttfonts.py:670: UserWarning: cmap value too big/small: -32808
  warnings.warn("cmap value too big/small: %s" % cm)
/Users/weiping/dev/Learn/langchain-ai/deep_agent_0530/.venv/lib/python3.13/site-packages/fpdf/ttfonts.py:670: UserWarning: cmap value too big/small: -33295
  warnings.warn("cmap value too big/small: %s" % cm)
/Users/weiping/dev/Learn/langchain-ai/deep_agent_0530/.venv/lib/python3.13/site-packages/fpdf/ttfonts.py:670: UserWarning: cmap value too big/small: -33324
  warnings.warn("cmap value too big/small: %s" % cm)
/Users/weiping/dev/Learn/langchain-ai/deep_agent_0530/.venv/lib/python3.13/site-packages/fpdf/ttfonts.py:670: UserWarning: cmap value too big/small: -33613
  warnings.warn("cmap value too big/small: %s" % cm)
/Users/weiping/dev/Learn/langchain-ai/deep_agent_0530/.venv/lib/python3.13/site-packages/fpdf/ttfonts.py:670: UserWarning: cmap value too big/small: -34165
  warnin

✅ 成功生成企业级专业审计手册: GlobalCorp_Professional_Audit.pdf


## 2. 模拟 RAG (分块检索) 的局限
用户提问：**“CYBER-TIGER 项目在 APAC 地区违规，处罚标准是多少？”**

**RAG 的逻辑短板：**
由于文档非常大，RAG 可能检索到了包含 `CYBER-TIGER` 的封面定义，却漏掉了第 15 页深处的 `35%` 处罚条款，因为它无法将两个相隔 14 页的概念建立语义关联。

In [3]:
rag_context = """
[第 1 页 检索碎片]: 【CYBER-TIGER (网络虎计划)】：本代号专指所有涉及跨国界的数据迁移、数据库热备份及相关活动。
[第 8 页 检索碎片]: 内部会议纪要应在 48 小时内完成存档，确保合规性。
"""
query = "CYBER-TIGER 项目在 APAC 地区违规，具体的罚金标准是多少？"

print("--- 🚫 Vector RAG 回答 (模拟碎片信息) ---")
prompt = f"你是一个审计员。根据提供的上下文回答问题，若信息不足请直说不知道。\n\n上下文：\n{rag_context}\n\n问题：{query}"
print(model.invoke([HumanMessage(content=prompt)]).content)

--- 🚫 Vector RAG 回答 (模拟碎片信息) ---
不知道


## 3. 实现 KV / CAG 模式 (全文逻辑穿透)
我们一次性读取 PDF 的全部 15 页内容送入 Agnes 模型。模型将展现其惊人的关联推理能力。

In [4]:
def read_pdf_full(path):
    reader = PdfReader(path)
    return "\n".join([p.extract_text() for p in reader.pages])

full_manual = read_pdf_full("GlobalCorp_Professional_Audit.pdf")

print("--- ✅ KV / CAG 回答 (全文深度分析) ---")
prompt_cag = f"你是一个 GlobalCorp 高级分析师。请阅读整份规程，关联不同章节的定义和处罚规则，回答用户问题。\n\n手册全文：\n{full_manual}\n\n问题：{query}"
print(model.invoke([HumanMessage(content=prompt_cag)]).content)

--- ✅ KV / CAG 回答 (全文深度分析) ---
根据规程第 1 章第 1.1 节，CYBER-TIGER（网络虎计划）涉及“云端数据迁移”及“数据库热备份”等活动。

根据规程第 99 章第 99.4 节关于跨境业务特别违规处理的规定，若此类活动在亚太地区（APAC）分支机构存在未授权操作，罚金标准为：**扣除该项目总预算的 35% 作为合规罚金**。


## 4. 复盘：为什么我们需要 Graphiti？

通过上面的对比，我们发现：
1. **KV/CAG 完美解决了静态知识**：对于 15-50 页的文档，它是当前的“最优解”。
2. **但 Alex 面临着更大的问题**：
    - 随着讨论的深入，对话记录也会变得像这份 15 页的手册一样臃肿。
    - Alex 会在对话中不断修正事实（例如：“CYBER-TIGER 项目暂时停工”）。
    - 如果只是简单的把所有对话一直堆叠在 KV 库中，AI 很快就会陷入“注意力迷雾”。

**下一章预告：**
我们将引入 **Graphiti**。它的作用不再是处理这份静态 PDF，而是作为 Alex 和 AI 对话的 **“动态归档器”**。它会将对话中的每一个关键决策（如项目状态、人员变动）提取为图谱事实，从而管理无限增长的对话上下文。